# <font color = 'red'> DEPENDENCIAS

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.preprocessing import MinMaxScaler

import sys
import os

# Agregar la carpeta calibration_code al path
sys.path.append(os.path.abspath("../../calibration_code"))

# Ahora puedes importar los módulos personalizados
from modelling_tools import (plot_histogram, plot_univariate_freq, assign_deciles, count_categories_by_decile, 
                             calculate_category_proportions, summarize_decile_analysis, summarize_grouped_deciles, group_deciles,
                             compute_odds_ratio)
from visualization_tools import plot_interactive_chart
from utils import g
from config import get_data_path, get_code_path
from data_cleaning import check_dataframe_quality

# <font color = 'red'> CARGA DE DATOS

In [2]:
df = pd.read_csv(get_data_path("bivariate_preprocessed_data.csv"))

In [3]:
res = check_dataframe_quality(df)

No missing values found.
No infinite values found.
No duplicate rows found.


# <font color = 'red'> ANÁLISIS

In [4]:
col = "Net_Salary_Ratio"

## <font color = 'skyblue'> ANÁLISIS GENERAL

La mediana de los ingresos de los clientes malos no parece significativamente diferente a los clientes Standard y a los Buenos.

In [7]:
fig_box = px.box(df, x="Credit_Mix", y=col, title=f"Distribution of {col} by Credit Score Category")
fig_box.show()

## <font color = 'skyblue'> ANÁLISIS POR DECILES

In [8]:
continuous_variable= col
decile_col_name = continuous_variable + '_Decile'
target_col_string = "Credit_Mix" # variable dependiente con nombres string
target_col = 'Credit_Score' # variable dependiente int (para modelos)

In [9]:
analysis_summary = summarize_decile_analysis(df, continuous_variable, decile_col_name, target_col_string)

# Obtener los resultados
df_deciles = analysis_summary["df_deciles"]  # DataFrame con los deciles asignados
deciles_summary = analysis_summary["decile_summary"]  # Resumen de deciles con conteos y proporciones
display(deciles_summary)
res = check_dataframe_quality(df_deciles)

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_Bad,prop_Good,prop_Standard
Net_Salary_Ratio_Decile,,,,,,,,,,
0,4.313301,7.433533,10007,0.10007,3467,1854,4686,0.346457,0.185270,0.468272
1,7.434605,7.871301,9994,0.09994,2464,2845,4685,0.246548,0.284671,0.468781
2,7.871540,8.071038,10003,0.10003,2521,2665,4817,0.252024,0.266420,0.481556
3,8.071180,8.199846,10003,0.10003,1818,3657,4528,0.181745,0.365590,0.452664
4,8.199879,8.321629,9995,0.09995,1590,4059,4346,0.159080,0.406103,0.434817
5,8.321636,8.445280,10004,0.10004,1658,4170,4176,0.165734,0.416833,0.417433
6,8.445309,8.574042,9999,0.09999,1409,4119,4471,0.140914,0.411941,0.447145
7,8.574083,8.772369,9996,0.09996,2515,2781,4700,0.251601,0.278211,0.470188
8,8.772538,9.191763,10000,0.10000,2548,2849,4603,0.254800,0.284900,0.460300


No missing values found.
No infinite values found.
No duplicate rows found.


In [11]:
df[(df[continuous_variable] >= 8.199879) & (df[continuous_variable] <= 8.321629) & (df['Credit_Score'] == 2)].shape

(4059, 85)

El comportamiento no es el esperado para esta variable. Es mejor no utilizarla:



In [12]:
chart_types = {
    "prop_Good":"line",
    "prop_Standard":"line",
    "prop_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=deciles_summary,  
    y_columns=["prop_Bad", "prop_Good", "prop_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title=f"Proportion of Credit Score Categories by {continuous_variable}",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_Bad", "prop_Good", "prop_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_Bad": "red", "prop_Good":"lightgreen",
    "prop_Standard":"brown", "Decile_Count": "gray"}  
)

fig.show()